This notebook documents how the feature translation was built and tested,
one group at a time, against a sample book. The functions were then moved to App/features.py, which is the version the web app imports. 
This notebook is a record of the process, not the code that runs in production.

In [22]:
import pandas as pd
import numpy as np
import joblib
import re

In [23]:
#Chck if model loads in new file
bundle = joblib.load('../Model/book_rating_random_forest_model.joblib')
for k in bundle:
    print(k)

model
ordinal_encoder
language_groups
feature_columns
author_freq_map
publisher_freq_map


In [24]:
#Extracting a book to try our prediction on
ref = pd.read_csv('../Data/Processed/clean_books_with_text.csv')
ref[(ref['title'] == "The Hobbit or There and Back Again") &
    (ref['authors'] == "J.R.R. Tolkien")][
    ['title', 'authors', 'publisher', 'num_pages', 'language_code',
     'publication_year', 'ratings_count', 'text_reviews_count', 'average_rating']]

,title,authors,publisher,num_pages,language_code,publication_year,ratings_count,text_reviews_count,average_rating
1694,The Hobbit or There and Back Again,J.R.R. Tolkien,Houghton Mifflin,366,eng,2002,2530894,32871,4.27


In [25]:
#Dictionary with the features used to identify the book
sample = {
    'title': "The Hobbit or There and Back Again",
    'authors': "J.R.R. Tolkien",
    'publisher': "Houghton Mifflin",
    'num_pages': 366,
    'language_code': "eng",
    'publication_year': 2002,
    'ratings_count': 2530894,
    'text_reviews_count': 32871,
}

In [26]:
#Defining a function for all title features
def title_features(title):
    return {
        'title_chars':      len(title),
        'has_subtitle':     int(":" in title),
        'title_has_number': int(bool(re.search(r'\d', title))),
        'is_collection':    int(bool(re.search(
                              r'boxed set|box set|omnibus|collection|anthology',
                              title, re.I))),
    }

print(title_features(sample['title']))

{'title_chars': 34, 'has_subtitle': 0, 'title_has_number': 0, 'is_collection': 0}


In [27]:
#Defining a function for all author features
def author_features(authors, num_pages):
    num_authors = authors.count("/") + 1
    return{
        'num_authors': num_authors,
        'multiple_authors' : int(num_authors > 1),
        'pages_per_author' : num_pages / num_authors
    }
print(author_features(sample['authors'],sample['num_pages']))

{'num_authors': 1, 'multiple_authors': 0, 'pages_per_author': 366.0}


In [28]:
#Defining a function for Audiobook
def audiobook_feature(title, publisher):
    pub_kw   = r'audio|tantor|listening library|recorded books|books on tape|highbridge|your coach'
    title_kw = r'audio|unabridged|abridged|\bcd\b|audiobook|spoken|cassette'
    return {
        'is_audiobook' : int(bool(re.search(title_kw, title, re.I)) or
                   bool(re.search(pub_kw, publisher, re.I)))
    }
print(audiobook_feature(sample['title'], sample['publisher']))


{'is_audiobook': 0}


In [29]:
#Defining a function for Arithemtic features
def rating_features(ratings_count, text_reviews_count):
    log_ratings = float(np.log1p(ratings_count))
    rev_per_rating = text_reviews_count / ratings_count if ratings_count > 0 else 0
    return {
    'log_ratings':  log_ratings,
    'rev_per_rating': rev_per_rating
        }
print(rating_features(sample['ratings_count'],sample['text_reviews_count']))

{'log_ratings': 14.744083553087888, 'rev_per_rating': 0.012987900718086177}


In [30]:
#Defining a function for the frequency and ordinal features
def lookup_features(authors, publisher, bundle):
    a_ord, p_ord = bundle['ordinal_encoder'].transform(
        pd.DataFrame([[authors, publisher]], columns=['authors', 'publisher']))[0]
    return {
        'author_freq':    bundle['author_freq_map'].get(authors, 1),
        'publisher_freq': bundle['publisher_freq_map'].get(publisher, 1),
        'author_ord':     float(a_ord),
        'publisher_ord':  float(p_ord),
    }

print(lookup_features(sample['authors'], sample['publisher'], bundle))

{'author_freq': 17, 'publisher_freq': 19, 'author_ord': 2625.0, 'publisher_ord': 979.0}


In [31]:
#Defining a function for the language features
LANG_GROUPS = ['asian', 'classical', 'english', 'other', 'western_european']

def language_features(language_code, bundle):
    lang = bundle['language_groups'].get(language_code, 'other')
    return {f'language_group_{g}': int(lang == g) for g in LANG_GROUPS}

print(language_features(sample['language_code'], bundle))

{'language_group_asian': 0, 'language_group_classical': 0, 'language_group_english': 1, 'language_group_other': 0, 'language_group_western_european': 0}


In [32]:
#Combine all feature functions into the final model input
def build_features(title, authors, publisher, num_pages, language_code,
                   publication_year, ratings_count, text_reviews_count, bundle):
    feats = {'num_pages': num_pages, 'publication_year': publication_year}
    feats |= title_features(title)
    feats |= author_features(authors, num_pages)
    feats |= audiobook_feature(title, publisher)
    feats |= rating_features(ratings_count, text_reviews_count)
    feats |= lookup_features(authors, publisher, bundle)
    feats |= language_features(language_code, bundle)
    return pd.DataFrame([feats])[bundle['feature_columns']]

row = build_features(**sample, bundle=bundle)
pred = bundle['model'].predict(row)[0]
print(f"Predicted rating: {pred:.2f}")

Predicted rating: 4.24


In [33]:
#Testing features from feature.py
import sys
sys.path.append('../App')
from features import build_features

row = build_features(**sample, bundle=bundle)
print(f"{bundle['model'].predict(row)[0]:.2f}")

4.24
